In [1]:
!nvcc --version
!pip install git+https://github.com/afnan47/cuda.git
%load_ext nvcc_plugin

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-n9rcmi_i
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-n9rcmi_i
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4290 sha256=4e9908e68b388f0e102e73a0e012b145bb165d50d7ee86bee13b67376ed52393
  Stored in directory: /tmp/pip-ephem-wheel-cache-kc6a439p/wheels/e8/cf/c3/c90ca0d0bba7969f9b8670f5624f76d097123d656355c77053
Successfully built NVCCPlugin
created output directory at /content/src
Out bin /content/result.out


In [2]:
%%cu
#include <iostream>
using namespace std;

__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}


/* -------- INITIALIZE VECTOR -------- */
void initialize(int* vector, int size) {
    for (int i = 0; i < size; i++) {
        vector[i] = rand() % 10;
    }
}

/* -------- PRINT VECTOR -------- */
void print(int* vector, int size) {
    for (int i = 0; i < size; i++) {
        cout << vector[i] << " ";
    }
    cout << endl;
}

/* -------- MAIN FUNCTION -------- */
int main() {
    int N = 4;
    int vectorSize = N;
    size_t vectorBytes = vectorSize * sizeof(int);

    int *A, *B, *C;

    // Allocate host memory
    A = new int[vectorSize];
    B = new int[vectorSize];
    C = new int[vectorSize];

    // Initialize vectors
    initialize(A, vectorSize);
    initialize(B, vectorSize);

    cout << "Vector A: ";
    print(A, N);

    cout << "Vector B: ";
    print(B, N);

    // Device pointers
    int *X, *Y, *Z;

    // Allocate GPU memory
    cudaMalloc(&X, vectorBytes);
    cudaMalloc(&Y, vectorBytes);
    cudaMalloc(&Z, vectorBytes);

    // Copy data from CPU to GPU
    cudaMemcpy(X, A, vectorBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, vectorBytes, cudaMemcpyHostToDevice);

    // Define threads and blocks
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // Launch kernel
    add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, N);

    // Copy result back to CPU
    cudaMemcpy(C, Z, vectorBytes, cudaMemcpyDeviceToHost);

    cout << "Addition: ";
    print(C, N);

    // Free memory
    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Vector A: 3 6 7 5 
Vector B: 3 5 6 2 
Addition: 6 11 13 7 



In [ ]:
# Retrieving Notebook summary.

# Read [](file:///c%3A/Users/ADMIN/OneDrive/Desktop/hpc%20final/cuda_vector.ipynb#1-1), lines 1 to 10

# The selected code in your notebook (cuda_vector.ipynb, lines 2-4) is identical to the setup code in your cuda_matrix.ipynb notebook. For completeness, I'll provide the same detailed line-by-line explanation here. These are shell commands and Jupyter magic commands that prepare the environment for CUDA programming in Jupyter.

# ### Line 1: `!nvcc --version`
# - **What it does**: This is a shell command (prefixed with `!` to run it in the system shell from within Jupyter) that checks and displays the version of `nvcc`, which is the NVIDIA CUDA Compiler (nvcc). It queries the installed CUDA toolkit and prints details like the version number, build date, and copyright information.
# - **Why it's needed**: CUDA programming requires the NVIDIA CUDA toolkit to compile and run GPU-accelerated code. This command verifies that CUDA is installed and accessible. If CUDA isn't installed or the path isn't set up correctly, this will fail (e.g., "nvcc: command not found"). It's a common first step in CUDA setups to ensure compatibility.
# - **Output example**: Something like `nvcc: NVIDIA (R) Cuda compiler driver Copyright (c) 2005-2023 NVIDIA Corporation Built on Wed_Nov_22_10:17:15_PST_2023 Cuda compilation tools, release 12.3, V12.3.107 Build cuda_12.3.r12.3/compiler.33567101_0`.
# - **Potential issues**: If you're on a system without NVIDIA GPUs or CUDA drivers, this might not work. On Windows, ensure CUDA is installed via the NVIDIA installer and that the PATH environment variable includes the CUDA bin directory (e.g., `C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.3\bin`).
# - **Execution context**: This runs in the terminal/shell, not in Python. In Jupyter, the `!` prefix executes it synchronously.

# ### Line 2: `!pip install git+https://github.com/afnan47/cuda.git`
# - **What it does**: This is another shell command that uses `pip` (Python's package installer) to install a Python package directly from a GitHub repository. It clones the repo at `https://github.com/afnan47/cuda.git` and installs it as a package. The `git+` prefix tells pip to treat it as a Git repository source.
# - **Why it's needed**: The repository likely contains a custom CUDA-related Python package or extension (possibly a wrapper or plugin for CUDA in Python/Jupyter). This is often used for tools like `nvcc_plugin` (see line 3), which allows running CUDA code in Jupyter cells. Without this, the extension in line 3 won't load.
# - **Details on the repo**: `afnan47/cuda` appears to be a fork or custom repo providing CUDA utilities. It might include Python bindings or Jupyter integrations for CUDA. Installing from Git means you're getting the latest (or specific) version, but it could be unstable compared to PyPI packages.
# - **Potential issues**: 
#   - Requires Git to be installed (for cloning the repo).
#   - Might fail if the repo is private, doesn't exist, or has dependencies (e.g., NumPy, CUDA runtime). Check for error messages like "Could not find a version that satisfies the requirement."
#   - On Windows, ensure pip is up-to-date (`pip install --upgrade pip`) and that you have permissions to install packages.
#   - This installs globally or in the current environment; if using virtual environments, activate it first.
# - **Execution context**: Runs in the shell via pip. It may take time to download and install, and you'll see progress output.

# ### Line 3: `%load_ext nvcc_plugin`
# - **What it does**: This is a Jupyter "magic command" (prefixed with `%`) that loads a Jupyter extension called `nvcc_plugin`. Extensions add functionality to Jupyter, and this one specifically enables compiling and running CUDA code directly in notebook cells using the `%%cu` magic (as seen in your next cell).
# - **Why it's needed**: Standard Jupyter doesn't support CUDA compilation. This extension integrates `nvcc` (from line 1) into Jupyter, allowing you to write CUDA kernels in cells and compile them on-the-fly. Without it, CUDA code cells (like `%%cu`) won't work.
# - **How it works**: After loading, you can use `%%cu` at the start of a cell to indicate CUDA code. The extension compiles it with nvcc and runs it on the GPU. It likely depends on the package installed in line 2.
# - **Potential issues**: 
#   - If the extension isn't installed (from line 2), this will fail with "No module named 'nvcc_plugin'" or similar.
#   - Requires a compatible CUDA setup (GPUs, drivers). On non-NVIDIA systems, it won't function.
#   - If Jupyter is running in an environment without the extension, restart the kernel after installation.
# - **Execution context**: This is a Jupyter-specific command that modifies the notebook's runtime environment. It loads once per session.

# ### Overall Context
# - **Purpose**: These lines set up a CUDA development environment in Jupyter, enabling GPU-accelerated vector operations (based on the notebook name cuda_vector.ipynb).
# - **Order matters**: Run them in sequence—check CUDA, install dependencies, load extension.
# - **Next steps**: Your second cell likely contains the main CUDA vector code, which this enables.

# If you meant the entire notebook or a different selection, let me know! If you want me to explain the vector code (second cell), I can do that next.


# Read [](file:///c%3A/Users/ADMIN/OneDrive/Desktop/hpc%20final/cuda_vector.ipynb#7-7), lines 7 to 93

# Below is a detailed, line-by-line explanation of the entire code in your notebook's second cell (lines 7–93). This is a complete CUDA C++ program for performing vector addition on a GPU. I'll break it down sequentially, explaining what each line or block does, its purpose, and key concepts. The code adds two vectors (A and B) element-wise to produce C, using random values for demonstration. It runs on the GPU via a kernel.

# ### Line 7: `%%cu`
# - **What it does**: Jupyter magic command indicating CUDA code follows. Tells `nvcc_plugin` to compile and run this as CUDA on the GPU.
# - **Why needed**: Enables CUDA in Jupyter.
# - **Details**: Code after this is C++ with CUDA extensions.

# ### Lines 8–9: `#include <iostream>` and `using namespace std;`
# - **What they do**: Includes I/O library and brings `std` namespace into scope for `cout`, etc.
# - **Why needed**: For printing vectors and using standard functions.
# - **Details**: Standard in C++ programs.

# ### Lines 11–18: `__global__ void add(int* A, int* B, int* C, int size) {` to `}`
# - **What they do**: Defines the CUDA kernel `add`.
#   - `__global__`: Marks it as a GPU kernel.
#   - Parameters: Pointers to int arrays (vectors) and size.
#   - `tid = blockIdx.x * blockDim.x + threadIdx.x;`: Calculates thread ID (1D index).
#   - `if (tid < size)`: Bounds check to avoid out-of-bounds access.
#   - `C[tid] = A[tid] + B[tid];`: Adds corresponding elements.
# - **Why needed**: Performs parallel addition; each thread handles one element.
# - **Details**: 1D grid; simple element-wise operation.

# ### Lines 20–27: `/* -------- INITIALIZE VECTOR -------- */` to `void print(int* vector, int size) {` to `}`
# - **What they do**: Helper functions.
#   - `initialize`: Fills vector with random ints (0-9) using `rand()`.
#   - `print`: Prints vector elements separated by spaces.
# - **Why needed**: Initializes data randomly and displays results.
# - **Details**: CPU functions; `rand()` from `<cstdlib>` (included via `<iostream>` indirectly).

# ### Lines 29–31: `/* -------- MAIN FUNCTION -------- */` and `int main() {` and `int N = 4;`
# - **What they do**: Starts `main`; sets vector size N to 4.
# - **Why needed**: Program entry; fixed small size for demo.
# - **Details**: N=4 means 4-element vectors.

# ### Lines 32–33: `int vectorSize = N;` and `size_t vectorBytes = vectorSize * sizeof(int);`
# - **What they do**: Sets vector size and calculates memory size in bytes.
# - **Why needed**: Prepares for memory allocation.
# - **Details**: `sizeof(int)` is typically 4 bytes.

# ### Lines 35–37: `int *A, *B, *C;` and host allocations
# - **What they do**: Declares pointers and allocates host memory with `new`.
# - **Why needed**: Stores vectors on CPU.
# - **Details**: Dynamic arrays for A, B, C.

# ### Lines 39–42: Initialize and print vectors
# - **What they do**: Calls `initialize` for A and B, then prints them.
# - **Why needed**: Sets up input data and shows it.
# - **Details**: Uses helper functions.

# ### Lines 44–46: Device pointers and allocations
# - **What they do**: Declares GPU pointers (X, Y, Z) and allocates GPU memory with `cudaMalloc`.
# - **Why needed**: GPU needs its own memory.
# - **Details**: Mirrors host pointers.

# ### Lines 48–50: Copy to GPU
# - **What they do**: Copies A and B from host to device using `cudaMemcpy`.
# - **Why needed**: Transfers inputs to GPU.
# - **Details**: `cudaMemcpyHostToDevice` direction.

# ### Lines 52–53: Threads and blocks
# - **What they do**: Calculates grid dimensions.
#   - `threadsPerBlock = 256`: Threads per block.
#   - `blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;`: Ensures enough blocks (ceils division).
# - **Why needed**: Defines CUDA launch configuration.
# - **Details**: For N=4, 1 block suffices.

# ### Line 55: Kernel launch
# - **What they do**: Launches `add` kernel with grid/block dims.
# - **Why needed**: Executes parallel addition.
# - **Details**: `<<<blocksPerGrid, threadsPerBlock>>>` syntax.

# ### Lines 57–59: Copy back and print
# - **What they do**: Copies result C from GPU to host, then prints it.
# - **Why needed**: Retrieves and displays output.
# - **Details**: `cudaMemcpyDeviceToHost`.

# ### Lines 61–67: Free memory
# - **What they do**: Deallocates host (`delete[]`) and device (`cudaFree`) memory.
# - **Why needed**: Prevents leaks.
# - **Details**: Essential cleanup.

# ### Lines 69–70: `return 0;` and `}`
# - **What they do**: Exits successfully.
# - **Details**: Standard.

# ### Overall Notes
# - **Flow**: CPU init/print → GPU alloc/copy → kernel → copy back/print → free.
# - **CUDA Basics**: 1D kernel for vectors; parallel per element.
# - **Performance**: Small N=4; scales to millions in real use.
# - **Output**: Random A/B, C = A+B element-wise.

# If you run this and need help, or want the setup explained again, let me know!



In [ ]:
# Below is a list of expected viva (oral examination) questions and answers based on the CUDA vector addition code in your cuda_vector.ipynb notebook. These are typical questions that might be asked in a high-performance computing (HPC) or parallel programming course, focusing on CUDA concepts, the code's implementation, and related theory. I've drawn directly from the code's structure (e.g., kernel, memory management, thread organization) and general CUDA principles. Answers are concise but detailed, assuming a basic understanding of C++ and parallel computing.

# ### 1. **What is CUDA, and why is it used in this code?**
#    - **Answer**: CUDA (Compute Unified Device Architecture) is NVIDIA's parallel computing platform and programming model for GPU-accelerated computing. It's used here to perform vector addition on the GPU for speed, as GPUs excel at parallel tasks like element-wise operations on large datasets. The code offloads the addition kernel to the GPU, leveraging thousands of threads for parallelism, which is faster than CPU serial execution for large vectors.

# ### 2. **Explain the role of the `__global__` keyword in the `add` kernel.**
#    - **Answer**: `__global__` declares the `add` function as a CUDA kernel that runs on the GPU but is called from the CPU. It indicates the function is executed in parallel across multiple GPU threads. In the code, `add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, N);` launches it with specified grid and block dimensions.

# ### 3. **How does the code calculate the thread ID (`tid`) in the kernel, and why is the bounds check `if (tid < size)` necessary?**
#    - **Answer**: `tid = blockIdx.x * blockDim.x + threadIdx.x;` computes a unique 1D index for each thread based on its block and thread position in the grid. The bounds check prevents out-of-bounds memory access, as the total threads (blocks * threads per block) might exceed the vector size (N=4). Without it, threads beyond N could cause errors or undefined behavior.

# ### 4. **Describe the memory management in this code. What are host and device memory, and how are they handled?**
#    - **Answer**: Host memory (CPU RAM) stores data like vectors A, B, C using `new` (e.g., `A = new int[vectorSize];`). Device memory (GPU RAM) is allocated with `cudaMalloc` (e.g., `cudaMalloc(&X, vectorBytes);`). Data is copied from host to device via `cudaMemcpy` with `cudaMemcpyHostToDevice`, processed on GPU, and copied back with `cudaMemcpyDeviceToHost`. Memory is freed with `delete[]` for host and `cudaFree` for device to avoid leaks.

# ### 5. **Why is `cudaMemcpy` used twice in the code, and what are the implications of not using it correctly?**
#    - **Answer**: It's used to transfer data: first to copy inputs (A, B) to GPU (`cudaMemcpyHostToDevice`), then to copy the result (C) back to CPU (`cudaMemcpyDeviceToHost`). Incorrect usage (e.g., wrong direction or size) could lead to data corruption, crashes, or silent errors. The code ensures bidirectional transfer for GPU computation.

# ### 6. **Explain the grid and block configuration in the code. How are `threadsPerBlock` and `blocksPerGrid` calculated?**
#    - **Answer**: `threadsPerBlock = 256;` sets 256 threads per block (a common optimal value for occupancy). `blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;` calculates blocks needed, using ceiling division to cover all elements (e.g., for N=4, 1 block). This 1D grid launches enough threads for parallel vector processing, balancing workload and GPU resources.

# ### 7. **What is the purpose of the `%%cu` magic command in the Jupyter cell?**
#    - **Answer**: It tells the `nvcc_plugin` extension to compile and run the following code as CUDA C++ on the GPU. Without it, the code would be treated as Python, causing compilation errors. It's essential for integrating CUDA into Jupyter notebooks.

# ### 8. **How does the code initialize vectors, and why use `rand() % 10`?**
#    - **Answer**: The `initialize` function fills vectors with random integers from 0-9 using `rand() % 10`. This provides varied test data for demonstration. `rand()` generates pseudo-random numbers; `% 10` limits the range. It's called for A and B before GPU processing to simulate real input data.

# ### 9. **Compare vector addition on CPU vs. GPU in this context. Why is GPU preferred for large vectors?**
#    - **Answer**: CPU does it serially (one element at a time), while GPU does it in parallel (one thread per element). For small N=4, CPU might be faster due to overhead, but for large vectors (e.g., millions), GPU scales better with thousands of cores. The code demonstrates GPU parallelism, reducing time from O(N) serial to near-constant parallel time.

# ### 10. **What happens if the vector size N is larger than the total threads launched? How does the code handle it?**
#     - **Answer**: If N > total threads, some elements won't be processed. The kernel's `if (tid < size)` ensures only valid indices are handled, preventing errors. For N=4 and 256 threads/block, it's fine, but for larger N, more blocks are needed (as calculated).

# ### 11. **Explain the output of the code. What would be an example for N=4?**
#     - **Answer**: It prints vectors A and B (random 0-9), then C (A+B element-wise). Example: A=[1,2,3,4], B=[5,6,7,8] → C=[6,8,10,12]. This verifies correctness. The code uses `cout` and `print` for formatted output.

# ### 12. **What are potential errors in this code, and how can they be fixed?**
#     - **Answer**: Common issues: CUDA not installed (check with `!nvcc --version`), memory allocation failures (add error checks like `if (cudaMalloc(...) != cudaSuccess)`), or kernel launch errors. Fix by ensuring NVIDIA drivers, correct paths, and adding `cudaDeviceSynchronize()` after kernel for debugging. For large N, increase blocks.

# ### 13. **How does this code demonstrate parallelism? Relate it to SIMD or MIMD.**
#     - **Answer**: It uses data parallelism (SIMD-like), where each thread performs the same operation (addition) on different data elements. CUDA is MIMD at the grid level but SIMD within warps (32 threads). The kernel assigns one thread per vector element, exploiting GPU's parallel architecture.

# ### 14. **Why is `sizeof(int)` used in `vectorBytes`, and what if the data type changed?**
#     - **Answer**: It calculates bytes per element (4 for int). If changed to float, use `sizeof(float)` (4) or double (8). This ensures correct memory allocation; mismatched sizes cause copy errors or corruption.

# ### 15. **What is the significance of `cudaFree` and `delete[]` at the end?**
#     - **Answer**: They deallocate memory to prevent leaks, which are critical in GPU programming where resources are limited. `cudaFree` releases GPU memory, `delete[]` releases CPU memory. Omitting them could crash large programs or exhaust memory.

# If you have a specific focus (e.g., more on kernels or memory), or need answers for the matrix code, let me know! Practice explaining these verbally for your viva.